# CSEN1095 Lab 3 — Task 1: Inconsistencies and duplicates

**Abdullah Essam · 58-0340 · MET**

This notebook profiles the supplied submissions, cleans a copy with pandas, removes duplicates using the assignment rules, checks the canonical values and key, and saves `club_signups_clean.csv`. The original `club_signups.csv` is never edited.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd()
source_path = DATA_DIR / "club_signups.csv"
output_path = DATA_DIR / "club_signups_clean.csv"
raw = pd.read_csv(source_path)
df = raw.copy()
print(f"Loaded {source_path.name}: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded club_signups.csv: 39 rows, 9 columns


## 1. First look

Show the shape, dtypes, and counts for each categorical column before changing any values.

In [2]:
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
categorical_columns = ["faculty", "city", "club", "fee_paid"]
for column in categorical_columns:
    print(f"\n{column} value counts:")
    print(df[column].value_counts(dropna=False).to_string())

Shape: (39, 9)

Dtypes:
signup_id       object
student_id      object
full_name       object
email           object
faculty         object
city            object
club            object
signed_up_at    object
fee_paid        object
dtype: object

faculty value counts:
faculty
Pharmacy                                  7
EMS                                       7
IET                                       5
Management                                4
Information Engineering and Technology    4
MET                                       3
iet                                       3
Engineering and Materials Science         1
Pharma                                    1
met                                       1
Media Engineering and Technology          1
pharmacy                                  1
ems                                       1

city value counts:
city
Cairo         13
Alexandria    11
Giza           4
Alex           4
cairo          2
giza           2
alexandria     1
CAIRO   

## 2. Standardize values and parse timestamps

Normalize whitespace and case first, then explicitly map the remaining variants. The slash dates are day/month: `13/09/2026` cannot be month/day. The two layouts are ISO `YYYY-MM-DD HH:MM` and slash-separated `DD/MM/YYYY HH:MM`.

In [3]:
faculty_map = {
    "met": "MET",
    "media engineering and technology": "MET",
    "information engineering and technology": "MET",
    "iet": "IET",
    "ems": "EMS",
    "engineering and materials science": "EMS",
    "pharmacy": "Pharmacy",
    "pharma": "Pharmacy",
    "management": "Management",
}
club_map = {
    "robotics": "Robotics",
    "debate": "Debate",
    "debating": "Debate",
    "debate club": "Debate",
    "music": "Music",
    "football": "Football",
    "soccer": "Football",
    "chess": "Chess",
    "chess club": "Chess",
}
city_map = {
    "cairo": "Cairo",
    "giza": "Giza",
    "el giza": "Giza",
    "alexandria": "Alexandria",
    "alex": "Alexandria",
}

def normalize_and_map(series, mapping):
    normalized = series.astype("string").str.strip().str.lower()
    mapped = normalized.map(mapping)
    if mapped.isna().any():
        raise ValueError(f"Unmapped variants in {series.name}: {normalized[mapped.isna()].unique()}")
    return mapped

df["faculty"] = normalize_and_map(df["faculty"], faculty_map)
df["club"] = normalize_and_map(df["club"], club_map)
df["city"] = normalize_and_map(df["city"], city_map)

df["full_name"] = (
    df["full_name"].astype("string").str.split().str.join(" ").str.title()
)
df["email"] = df["email"].astype("string").str.strip().str.lower()

fee_map = {
    "yes": True, "y": True, "true": True, "1": True,
    "no": False, "n": False, "false": False, "0": False,
}
fee_normalized = df["fee_paid"].astype("string").str.strip().str.lower()
df["fee_paid"] = fee_normalized.map(fee_map)
assert df["fee_paid"].notna().all(), "Unexpected fee_paid value found"
df["fee_paid"] = df["fee_paid"].astype(bool)

slash_dates = df["signed_up_at"].astype("string").str.contains("/", regex=False)
slash_parts = df.loc[slash_dates, "signed_up_at"].str.split("/", expand=True)
assert (slash_parts[0].astype(int) > 12).any()
assert (slash_parts[1].astype(int) <= 12).all()
df["signed_up_at"] = pd.to_datetime(
    df["signed_up_at"], format="mixed", dayfirst=True, errors="raise"
)

allowed = {
    "faculty": {"MET", "IET", "EMS", "Pharmacy", "Management"},
    "club": {"Robotics", "Debate", "Music", "Football", "Chess"},
    "city": {"Cairo", "Giza", "Alexandria"},
}
for column, values in allowed.items():
    assert set(df[column].dropna().unique()) <= values
assert df["fee_paid"].dtype == bool
assert df["full_name"].str.contains(r"\s{2,}").sum() == 0
assert df["email"].eq(df["email"].str.strip().str.lower()).all()
print("Canonical values and types checked.")
print("Parsed timestamp range:", df["signed_up_at"].min(), "to", df["signed_up_at"].max())

Canonical values and types checked.
Parsed timestamp range: 2026-09-13 14:36:00 to 2026-09-18 23:36:00


## 3. Remove duplicates

Count exact copies after standardization. Then compare duplicate counts before and after canonicalizing the club spelling, and keep the latest submission for each `student_id` + `club`.

In [4]:
exact_duplicate_count = int(df.duplicated().sum())
print("Exact duplicate rows:", exact_duplicate_count)
df = df.drop_duplicates().copy()
assert len(df) == len(raw) - exact_duplicate_count

# What a student_id + raw club dedupe would detect after exact-copy removal.
raw_after_exact = raw.drop_duplicates().copy()
raw_club_duplicate_rows = int(raw_after_exact.duplicated(["student_id", "club"]).sum())
canonical_club_duplicate_rows = int(df.duplicated(["student_id", "club"]).sum())
print("Repeated student/club rows before club spelling cleanup:", raw_club_duplicate_rows)
print("Repeated student/club rows after club spelling cleanup:", canonical_club_duplicate_rows)

before_pair_dedupe = len(df)
df = df.sort_values("signed_up_at", kind="stable")
df = df.drop_duplicates(subset=["student_id", "club"], keep="last").copy()
repeat_signup_removed = before_pair_dedupe - len(df)
print("Resubmission rows removed (latest kept):", repeat_signup_removed)
print("Rows remaining:", len(df))
assert not df.duplicated(["student_id", "club"]).any()
assert len(df) == 32

Exact duplicate rows: 3
Repeated student/club rows before club spelling cleanup: 2
Repeated student/club rows after club spelling cleanup: 4
Resubmission rows removed (latest kept): 4
Rows remaining: 32


## 4. Save and inspect the clean table

The clean CSV contains one row per student per canonical club. The source file remains unchanged.

In [5]:
df.to_csv(output_path, index=False)
print(f"Saved {output_path.name}")
print("Clean shape:", df.shape)
print("Final dtypes:")
print(df.dtypes)
display(df.head(10))

Saved club_signups_clean.csv
Clean shape: (32, 9)
Final dtypes:
signup_id               object
student_id              object
full_name               object
email           string[python]
faculty                 object
city                    object
club                    object
signed_up_at    datetime64[ns]
fee_paid                  bool
dtype: object


signup_id,student_id,full_name,email,faculty,city,club,signed_up_at,fee_paid
S001,58-9118,Mariam Kamel,mariam.kamel@student.guc.edu.eg,IET,Giza,Robotics,2026-09-13 14:36:00,True
S002,55-2992,Mohamed Adel,mohamed.adel@student.guc.edu.eg,MET,Alexandria,Debate,2026-09-13 15:38:00,True
S003,55-9245,Ali Ibrahim,ali.ibrahim@student.guc.edu.eg,MET,Cairo,Chess,2026-09-13 21:16:00,True
S004,61-7344,Laila Mostafa,laila.mostafa@student.guc.edu.eg,IET,Alexandria,Chess,2026-09-13 22:36:00,True
S005,61-4733,Omar Adel,omar.adel@student.guc.edu.eg,EMS,Cairo,Football,2026-09-13 23:52:00,True
S006,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Football,2026-09-14 01:22:00,True
S007,61-2495,Ahmed Adel,ahmed.adel@student.guc.edu.eg,Pharmacy,Alexandria,Robotics,2026-09-14 17:03:00,True
S008,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Music,2026-09-14 23:32:00,True
S009,61-4844,Mohamed Adel,mohamed.adel@student.guc.edu.eg,EMS,Cairo,Robotics,2026-09-15 01:05:00,True
S010,61-3846,Ahmed Zaki,ahmed.zaki@student.guc.edu.eg,Management,Alexandria,Chess,2026-09-15 03:12:00,False
